# Internal Error Analysis

Find validation failures that are shared across multiple model artifacts.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MIN_MODELS = 2

PROJECT_ROOT, ARTIFACTS_DIR

## Load Failure Files

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(str(text).split()).lower()


def make_failure_key(row: dict) -> str:
    payload = {
        "requirement": normalize_text(row.get("requirement", "")),
        "resume_snippet": normalize_text(row.get("resume_snippet", "")),
        "label": int(row.get("label", -1)),
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists() or path.stat().st_size == 0:
        return rows
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))
    return rows


failure_paths = sorted(ARTIFACTS_DIR.glob("*/validation_failures.jsonl"))
records = []
for path in failure_paths:
    model_name = path.parent.name
    for row in read_jsonl(path):
        row = dict(row)
        row["model"] = model_name
        row["artifact_dir"] = str(path.parent)
        row["failure_key"] = make_failure_key(row)
        records.append(row)

failures_df = pd.DataFrame(records)
print(f"failure files: {len(failure_paths)}")
print(f"failure rows: {len(failures_df)}")
failures_df.head()

## Failure Counts By Model

In [ ]:
if failures_df.empty:
    summary_df = pd.DataFrame(columns=["model", "num_failures"])
else:
    summary_df = (
        failures_df.groupby("model")
        .size()
        .reset_index(name="num_failures")
        .sort_values("num_failures", ascending=False)
    )

summary_df

## Common Failed Samples

In [ ]:
if failures_df.empty:
    common_failures_df = pd.DataFrame()
else:
    common_failures_df = (
        failures_df.groupby("failure_key")
        .agg(
            n_models=("model", "nunique"),
            models=("model", lambda values: sorted(set(values))),
            requirement=("requirement", "first"),
            label=("label", "first"),
            predictions=("prediction", lambda values: list(values)),
            probabilities=("probability", lambda values: [round(float(v), 4) for v in values]),
            resume_snippet=("resume_snippet", "first"),
        )
        .reset_index()
    )
    common_failures_df = common_failures_df[common_failures_df["n_models"] >= MIN_MODELS]
    common_failures_df = common_failures_df.sort_values(["n_models", "requirement"], ascending=[False, True])

common_failures_df

## Failed By Every Model

In [ ]:
all_models = sorted(failures_df["model"].unique()) if not failures_df.empty else []
failed_by_every_model_df = common_failures_df[common_failures_df["n_models"] == len(all_models)] if all_models else pd.DataFrame()

print(f"models considered: {len(all_models)}")
failed_by_every_model_df

## Pairwise Overlap

In [ ]:
def pairwise_overlap(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["model_a", "model_b", "shared_failures", "jaccard"])
    keys_by_model = {
        model: set(group["failure_key"])
        for model, group in df.groupby("model")
    }
    rows = []
    models = sorted(keys_by_model)
    for i, model_a in enumerate(models):
        for model_b in models[i + 1 :]:
            a = keys_by_model[model_a]
            b = keys_by_model[model_b]
            union = a | b
            rows.append(
                {
                    "model_a": model_a,
                    "model_b": model_b,
                    "shared_failures": len(a & b),
                    "jaccard": len(a & b) / len(union) if union else 0.0,
                }
            )
    return pd.DataFrame(rows).sort_values(["shared_failures", "jaccard"], ascending=[False, False])


pairwise_overlap_df = pairwise_overlap(failures_df)
pairwise_overlap_df

## Inspect One Shared Failure

In [ ]:
if common_failures_df.empty:
    pd.DataFrame()
else:
    selected_key = common_failures_df.iloc[0]["failure_key"]
    failures_df[failures_df["failure_key"] == selected_key].sort_values("model")[[
        "model",
        "requirement",
        "label",
        "prediction",
        "probability",
        "resume_snippet",
    ]]

## Optional Export

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "artifacts" / "error_analysis"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

common_failures_df.to_csv(EXPORT_DIR / "common_failures.csv", index=False)
pairwise_overlap_df.to_csv(EXPORT_DIR / "pairwise_overlap.csv", index=False)

EXPORT_DIR